# AIoT-DA L6: Multiple Linear Regression (Kaggle Boston Housing)
## 跨領域資料分析作業流程 (CRISP-DM) 完整實作與 ChatGPT 協作指引

---

### 📌 本單元學習重點
1. **ChatGPT 提示詞工程協作**：學習如何透過結構化 Prompt 逐步引導 ChatGPT 完成波士頓房地產需求分析、代碼生成、特徵篩選與估價模型。
2. **CRISP-DM 方法論標準流程**：遵循跨領域資料探勘 6 大標準步驟（Business Understanding $\rightarrow$ Data Understanding $\rightarrow$ Data Preparation $\rightarrow$ Modeling $\rightarrow$ Evaluation $\rightarrow$ Deployment）。
3. **資料前處理與特徵工程**：
   - 13 個結構性、社會經濟與環境特徵之清洗與尺度檢視。
   - 虛擬變數 (CHAS 查爾斯河景觀) 之編碼。
   - `train_test_split` (80/20 分割, random_state=42) 評估未見樣本泛化能力。
4. **多元線性迴歸基本假定與 VIF 檢定**：檢視常態性、殘差分佈、異質變異與共線性。
5. **模型評估指標意涵**：深入解析 $R^2$、MAE、MSE、RMSE 的數學意義與房價預估價值。
6. **過擬合與欠擬合判斷**：透過 5-Fold Cross-Validation 學習曲線診斷模型收斂狀態。
7. **5 大特徵選擇演算法對比 (Feature Selection Benchmark)**：Lasso (L1), Forward SFS, RFE, Correlation Ranking, Exhaustive Search。
8. **模型部署與序列化**：使用 `pickle` 與 `gzip` 完成生產環境壓縮保存與自動化推論。

## 💬 與 ChatGPT 協作對話紀錄 (Prompt Engineering Guide)

> **Step 1 Prompt (資料取得與基本探勘)**:
> *"solve Boston Housing multiple linear regression problem in kaggle using LR, applying the CRISP-DM steps. Load Boston Housing data from web URL https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv. Convert to pandas DataFrame and print summary."*

> **Step 2 Prompt (資料前處理與分割)**:
> *"Separate features (13 predictors) and target (MEDV in $1,000s). Split dataset into train and test with test_size=0.2 and random_state=42. Explain why train_test_split is necessary for assessing generalization."*

> **Step 3 & 4 Prompt (建模與評估指標)**:
> *"Train a LinearRegression model. Compute R2, MAE, MSE, and RMSE. Plot Actual vs Predicted values. Also plot a learning curve comparing training error vs test error to judge overfitting vs underfitting."*

> **Step 5 Prompt (5 大特徵選擇比較 1 to 6 特徵)**:
> *"Implement forward selection for 1 to 6 features based on minimizing RMSE and maximizing R2. Compare with Lasso, RFE, and Exhaustive search. Display results in a clean table."*

> **Step 6 Prompt (部署)**:
> *"Export the trained optimal model using pickle and gzip. Show how to reload and perform inference on a new residential tract."*

---
## 🏢 CRISP-DM Step 1: Business Understanding (商業理解)

- **商業目標**：建立波士頓都會區房地產價格估算模型，根據社區犯罪率 (CRIM)、每戶房間數 (RM)、學區師生比 (PTRATIO)、低收入人口比例 (LSTAT) 等 13 項因素，預測房屋中位價 (MEDV，以千美元計)。
- **利害關係人**：房地產投資人、都會規劃師、公產稅務估價師、購屋家庭。
- **成功準則**：測試集 $R^2 > 0.65$，RMSE 控制在 $5,000 以內，模型具可解釋性。

---
## 📊 CRISP-DM Step 2: Data Understanding (資料理解)

載入 Kaggle Boston Housing 資料集，確認資料維度、無缺失值與基本統計摘要。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

url = 'https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv'
df = pd.read_csv(url)
df.columns = [c.upper() for c in df.columns]

print(f"資料維度: {df.shape[0]} 列, {df.shape[1]} 行")
print("\n前 5 筆樣本：")
display(df.head())

print("\n缺失值檢視：")
print(df.isnull().sum())

print("\n關鍵特徵統計摘要：")
display(df[['RM', 'LSTAT', 'PTRATIO', 'CRIM', 'DIS', 'TAX', 'MEDV']].describe().round(2))

In [ ]:
# 散佈圖檢視主要特徵與目標房價 (MEDV) 的線性關係
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.regplot(x='RM', y='MEDV', data=df, ax=axes[0], color='#2563eb', scatter_kws={'alpha': 0.5})
axes[0].set_title('Rooms (RM) vs Price', fontweight='bold')

sns.regplot(x='LSTAT', y='MEDV', data=df, ax=axes[1], color='#ef4444', scatter_kws={'alpha': 0.5})
axes[1].set_title('Lower Status % (LSTAT) vs Price', fontweight='bold')

sns.regplot(x='PTRATIO', y='MEDV', data=df, ax=axes[2], color='#10b981', scatter_kws={'alpha': 0.5})
axes[2].set_title('Pupil-Teacher (PTRATIO) vs Price', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 🧹 CRISP-DM Step 3: Data Preparation (資料準備)

進行特徵矩陣與標籤分離，執行 80/20 訓練/測試集分割 (`random_state=42`)。

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('MEDV', axis=1)
y = df['MEDV']

# 80% 訓練集, 20% 測試集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"訓練集大小: {X_train.shape[0]} 筆 (80%)")
print(f"測試集大小: {X_test.shape[0]} 筆 (20%)")

---
## 🤖 CRISP-DM Step 4: Modeling (模型建置)

使用 Ordinary Least Squares (OLS) 訓練全特徵多元線性迴歸模型。

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(X_train, y_train)

print(f"模型截距 (Intercept b0): {regressor.intercept_:.4f}")
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': regressor.coef_
}).sort_values(by='Coefficient', ascending=False)
display(coef_df)

---
## 📈 CRISP-DM Step 5: Evaluation & Diagnosis (評估與診斷)

計算測試集指標：$R^2$、MAE、MSE、RMSE，並繪製殘差分佈與學習曲線。

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

y_pred = regressor.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R-squared (R2 Score):        {r2:.6f} (解釋 {r2*100:.2f}% 測試房價變異)")
print(f"Mean Absolute Error (MAE):   ${mae*1000:,.2f}")
print(f"Mean Squared Error (MSE):    {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse*1000:,.2f}")

In [ ]:
# 繪製實際值 vs 預測值
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred, color='#3b82f6', edgecolors='k', alpha=0.7, s=60, label='Test Samples')
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal Fit (y = x)')
plt.xlabel('Actual Price MEDV ($1k)')
plt.ylabel('Predicted Price MEDV ($1k)')
plt.title('Boston Housing: Actual vs Predicted Price', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### 🎯 前向特徵選擇 (Forward Selection 1 to 6 特徵)

In [ ]:
# 前向特徵選擇模擬
selected = []
fs_history = []

for k in range(1, 7):
    best_f = None
    best_rmse = np.inf
    best_r2 = -np.inf
    remaining = [f for f in X.columns if f not in selected]
    
    for f in remaining:
        m = LinearRegression().fit(X_train[selected + [f]], y_train)
        p = m.predict(X_test[selected + [f]])
        curr_rmse = np.sqrt(mean_squared_error(y_test, p))
        curr_r2 = r2_score(y_test, p)
        if curr_rmse < best_rmse:
            best_rmse = curr_rmse
            best_r2 = curr_r2
            best_f = f
            
    selected.append(best_f)
    fs_history.append({
        'Step': k,
        'Features (k)': len(selected),
        'Selected Features': list(selected),
        'RMSE ($1k)': round(best_rmse, 3),
        'R2 Score': round(best_r2, 6)
    })

df_fs_results = pd.DataFrame(fs_history)
display(df_fs_results)

---
## 🚀 CRISP-DM Step 6: Deployment (部署與序列化)

保存最優模型至 `boston_model.pkl` 與 `boston_model.pkl.gz`，並示範推論。

In [ ]:
import pickle
import gzip

optimal_features = selected[:4]
opt_model = LinearRegression().fit(X_train[optimal_features], y_train)

# 序列化儲存
with open('boston_model.pkl', 'wb') as f:
    pickle.dump({'model': opt_model, 'features': optimal_features}, f)
    
with gzip.open('boston_model.pkl.gz', 'wb') as f:
    pickle.dump({'model': opt_model, 'features': optimal_features}, f)

print("模型已成功序列化儲存至 boston_model.pkl 與 boston_model.pkl.gz！")

# 載入與推論示範
with gzip.open('boston_model.pkl.gz', 'rb') as f:
    deployed = pickle.load(f)

sample_input = pd.DataFrame([[12.0, 18.0, 9.0, 400.0]], columns=deployed['features'])
val_pred = deployed['model'].predict(sample_input)[0]
print(f"\n生產環境推論範例 (LSTAT=12%, PTRATIO=18, RAD=9, TAX=$400):")
print(f"預期房價中位數: ${val_pred * 1000:,.2f}")